# RecallFirst — reproduce every number in the paper

*Recall First, Then Let the Model Choose: Gemma 4 as a Bug-Localization Selector, Tested on Post-Cutoff Issues*

This notebook recomputes every table, exact test and figure of the paper from the released per-instance results.
It needs **no GPU and no model** — about a minute on CPU. The GPU scripts that produced the data are in `harness/`.

Code and data: https://github.com/HussamHaroon/RecallFirst

In [ ]:
# Locate the release: Kaggle dataset input, a cloned repo, or the current folder.
import os, glob, shutil, sys
cands = glob.glob("/kaggle/input/*/data/orderings_wholerepo.jsonl") + glob.glob("/kaggle/input/*/*/data/orderings_wholerepo.jsonl")
if cands:                                   # Kaggle: copy to a writable folder
    src = os.path.dirname(os.path.dirname(cands[0]))
    shutil.copytree(src, "/kaggle/working/recallfirst", dirs_exist_ok=True)
    os.chdir("/kaggle/working/recallfirst")
elif not os.path.exists("data/orderings_wholerepo.jsonl") and os.path.exists("../data/orderings_wholerepo.jsonl"):
    os.chdir("..")
sys.path.insert(0, os.getcwd())
print("release root:", os.getcwd()); print(sorted(os.listdir("data")))

## 1. The data at a glance
57 SWE-bench Lite instances (8 repositories), the controls on the same instances, and 120 post-cutoff
SWE-bench-Live issues. Field-by-field documentation: `DATA_CARD.md`.

In [ ]:
import json, collections
load = lambda p: [json.loads(l) for l in open(p)]
lite = [x for x in load("data/orderings_wholerepo.jsonl") if "ranks" in x]
ctrl = load("data/controls_probe57.jsonl")
live = [x for x in load("data/postcutoff_live120.jsonl") if "ranks" in x]
print("Lite:", len(lite), collections.Counter(x["repo"] for x in lite))
print("Controls:", collections.Counter(x["condition"] for x in ctrl))
print("Post-cutoff:", len(live), "issues,", len({x["repo"] for x in live}), "repos,",
      min(x["created_at"] for x in live), "to", max(x["created_at"] for x in live))

## 2. Tables of §4.1–4.4 (hit rates on identical instances)

In [ ]:
hit = lambda r, k: r is not None and 0 < r <= k
def rank_in(order, gold): return next((i for i, f in enumerate(order or [], 1) if f == gold), None)
def row(name, ranks, ks=(1, 5, 10)):
    n = len(ranks); print(f"{name:46s}" + "".join(f"  top-{k}: {sum(hit(r, k) for r in ranks):3d}/{n} ({100*sum(hit(r, k) for r in ranks)/n:3.0f}%)" for k in ks))
L = {x["instance"]: x for x in lite}; ids = sorted(L)
C = {(x["instance"], x["condition"]): x for x in ctrl}
print("§4.1 — ranking BM25's top-25 files")
for m in ("nami_bpt", "nami_bits", "qlm", "h_real", "bm25"):
    row(m, [L[i]["ranks"][m] for i in ids])
row("Gemma selects from the same 25", [C[(i, "bm25pool")]["rank"] for i in ids])
inpool = sum(C[(i, "bm25pool")]["gold_in_pool"] for i in ids)
print(f"random order of the 25 (expected): top-5 {inpool*5/25/57:.0%}")
print("\n§4.3 — union pool (BM25-25 ∪ dense-25)")
for m in ("bm25", "dense", "gemma_paths", "gemma_names"):
    row(m, [rank_in(L[i]["orders"][m], L[i]["gold"]) if m.startswith("gemma") else L[i]["ranks"][m] for i in ids])
print("\n§4.4 — controls (union pool)")
for c in ("union_empty", "union_mismatch", "union_seed1", "union_seed2"):
    row(c, [C[(i, c)]["rank"] for i in ids])
P = [C[(i, "probe57")] for i in ids]
print("\n§4.4 — issue-only probe (10 options):", {k: sum(p[k] for p in P) for k in ("correct", "dense_correct", "bm25_correct", "path_tiebreak_correct")})
print("\n§4.4 — post-cutoff (120)")
for m in ("bm25", "dense", "gemma_fill"):
    row(m, [x["ranks"][m] for x in live])

## 3. Every exact test (McNemar with +b/−c), Holm within families, bootstrap CIs

In [ ]:
import subprocess
print(subprocess.run([sys.executable, "analysis/controls_postcutoff_stats.py"], capture_output=True, text=True).stdout)

## 4. Figures

In [ ]:
print(subprocess.run([sys.executable, "analysis/make_figures.py"], capture_output=True, text=True).stdout)
from IPython.display import Image, display
for f in ("fig1_pool_and_selection", "fig2_hard_probe", "fig3_post_cutoff"):
    display(Image(f"figures/{f}.png", width=900))

## 5. Evaluate your own localizer
Write one JSON line per instance — `{"instance": "...", "order": ["path/a.py", ...]}` — and compare it pairwise with
every released method on the identical instances:

```
python -m nami.stats data/orderings_wholerepo.jsonl --add my_localizer.jsonl --baseline dense
```

For the memorization question, run it on the 120 post-cutoff instances and compare its gap to dense with the gap
on Lite (+19.3 points on Lite, +12.5 after the cutoff for Gemma-4-12B-it).